# AURA — Prediction API Demo

This notebook walks through the public prediction surface of
`inference/`:

- `PhishingDetector.load_production()` and `PhishingDetector.from_paths(...)`
- `PhishingDetector.predict(...)` and the `PredictionResult` dataclass
- `PhishingDetector.predict_batch(...)` with a custom threshold
- `PhishingDetector.predict_safe(...)` — errors as data
- Threshold behaviour (how `predicted_label` flips as `threshold` changes)
- Three-zone classification (`ConfidenceZone.SPAM/REVIEW/NOT_SPAM`)
- Calibrated probabilities via `calibrator_path=`
- `prediction_id` — pairing predictions with confirmations

It runs top-to-bottom with no external data files; all sample emails
are synthesised inline. Only the documented public surface of the
`inference` package is used (see the project `README.md`).

For online-learning see `demo_online_learning.ipynb`, for drift
monitoring see `demo_drift_monitor.ipynb`, for LLM auto-review see
`demo_auto_reviewer.py`.

## 1. Setup

Import the public API, configure `aura.inference` logging at INFO, and
load the production detector. If `AURA_MODELS_DIR` is unset we resolve
the repository's `./models` directory relative to this notebook.

In [46]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('aura.inference').setLevel(logging.INFO)

repo_root = Path.cwd()
for _ in range(4):
    if (repo_root / 'inference' / '__init__.py').exists():
        break
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault('AURA_MODELS_DIR', str((repo_root / 'models').resolve()))

from inference import PhishingDetector, PredictionResult, ValidationError

detector = PhishingDetector.load_production()
print('active model version:', detector.version)

INFO aura.inference: loading model version=v1_0 from C:\Users\Administrator\Documents\Projects\AURA - Adaptive User Risk Analyzer\AURA_Model\models\v1_0\production\phishing_detector_mlp_classifier.pkl


active model version: v1_0


## 2. Synthetic sample emails

Two small pools of realistic-looking phishing and legitimate emails.
Phishing templates include spoofed display names, urgency language,
and suspicious URLs; legitimate templates are newsletters and
personal mail. We use a fixed seed so the notebook is reproducible.

In [47]:
import random
rng = random.Random(42)

PHISH_POOL = [
    {
        'sender': '"PayPal Security" <service@paypa1-alerts.com>',
        'subject': 'URGENT: Your account will be suspended in 24 hours!!!',
        'body': 'Dear customer, we detected suspicious activity on your account. '
                'Please verify your credentials at http://paypa1-alerts.com/verify '
                'within 24 hours or your account will be locked!!!',
    },
    {
        'sender': '"Microsoft 365" <no-reply@ms-support-verify.net>',
        'subject': 'Action required: verify your identity immediately',
        'body': 'Your mailbox is over quota. Click http://ms-support-verify.net/login '
                'to re-authenticate NOW, otherwise incoming mail will bounce.',
    },
    {
        'sender': '"DHL Express" <tracking@dhl-delivery-notice.info>',
        'subject': 'Package delivery failed — reschedule within 12 hours',
        'body': 'Your DHL package could not be delivered. Confirm address at '
                'http://dhl-delivery-notice.info/track?id=982137 before it is returned.',
    },
    {
        'sender': '"Chase Bank" <alerts@secure-chase-online.com>',
        'subject': 'Unusual sign-in attempt detected — confirm now',
        'body': 'Final notice: update payment details at '
                'http://secure-chase-online.com/account/update to avoid service disruption.',
    },
    {
        'sender': '"HR Department" <hr.payroll@company-benefits.co>',
        'subject': 'Payroll update: click to review your new paystub',
        'body': 'HR has posted an updated paystub. Download here: '
                'http://company-benefits.co/hr/paystub.pdf — password is your employee ID.',
    },
]

LEGIT_POOL = [
    {
        'sender': '"Jane Doe" <jane.doe@gmail.com>',
        'subject': 'Lunch on Saturday?',
        'body': 'Hey, are you free for lunch Saturday around noon? Thinking of that '
                'ramen place on 5th. Let me know.',
    },
    {
        'sender': '"GitHub" <noreply@github.com>',
        'subject': 'Your weekly digest: 5 new repositories trending',
        'body': 'Here are the repositories trending this week in your languages. '
                'Visit github.com/trending to see the full list.',
    },
    {
        'sender': '"ACM Weekly" <newsletter@acm.org>',
        'subject': 'ACM TechNews — April edition',
        'body': 'This edition covers advances in compilers, a retrospective on Unix, '
                'and upcoming conference deadlines. Read online at acm.org/technews.',
    },
    {
        'sender': '"Mom" <mom@familymail.net>',
        'subject': 'Recipe for that soup you liked',
        'body': 'Hi sweetie, here is the lentil soup recipe I promised. Soak the '
                'lentils overnight, then simmer with carrots and celery for an hour.',
    },
    {
        'sender': '"Coursera" <no-reply@t.mail.coursera.org>',
        'subject': 'New course recommendation based on your interests',
        'body': 'Based on your recent activity we thought you might enjoy the course '
                '"Algorithms, Part II" from Princeton. No pressure — just a heads up.',
    },
]

samples = []
for p in PHISH_POOL:
    samples.append({**p, 'label': 1})
for l in LEGIT_POOL:
    samples.append({**l, 'label': 0})
rng.shuffle(samples)

print(f'generated {len(samples)} samples ('
      f'{sum(s["label"] for s in samples)} phishing, '
      f'{sum(1 - s["label"] for s in samples)} legitimate)')

generated 10 samples (5 phishing, 5 legitimate)


## 3. Single prediction — the `PredictionResult` shape

`detector.predict(sender, subject, body, threshold=...)` returns a
`PredictionResult` dataclass with these fields:

- `predicted_label` — `0` (legitimate) or `1` (phishing), derived from
  `phishing_probability >= threshold`
- `phishing_probability`, `legitimate_probability`
- `threshold` — the threshold actually used for this prediction
- `model_version` — which version produced the result
- `engineered_features` — the 15 numeric features that sit alongside
  the TF–IDF matrices in the model's input vector

In [48]:
phish_example = next(s for s in samples if s['label'] == 1)
legit_example = next(s for s in samples if s['label'] == 0)

def show(res: PredictionResult, title: str) -> None:
    print(f'--- {title} ---')
    print('predicted_label       :', res.predicted_label)
    print('phishing_probability  :', round(res.phishing_probability, 4))
    print('legitimate_probability:', round(res.legitimate_probability, 4))
    print('threshold             :', res.threshold)
    print('model_version         :', res.model_version)
    print('engineered_features   :')
    for k, v in res.engineered_features.items():
        print(f'  {k:30s} {v:.4f}')
    print()

show(detector.predict(phish_example['sender'], phish_example['subject'], phish_example['body']),
     'phishing example')
show(detector.predict(legit_example['sender'], legit_example['subject'], legit_example['body']),
     'legitimate example')

--- phishing example ---
predicted_label       : 1
phishing_probability  : 0.9994
legitimate_probability: 0.0006
threshold             : 0.75
model_version         : v1_0
engineered_features   :
  body_word_count                11.0000
  body_exclamation_count         0.0000
  email_local_length             6.0000
  name_email_consistency         1.0000
  body_url_density               9.0909
  body_url_count                 1.0000
  body_entropy                   4.1664
  email_digit_ratio              0.0000
  domain_entropy                 3.4058
  domain_length                  23.0000
  subject_entropy                4.0037
  body_avg_word_length           9.4545
  sender_name_exists             1.0000
  subject_exclamation_count      0.0000
  domain_vowel_consonant_ratio   0.8182

--- legitimate example ---
predicted_label       : 0
phishing_probability  : 0.0056
legitimate_probability: 0.9944
threshold             : 0.75
model_version         : v1_0
engineered_features   :
  bod

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


### `to_dict()` — the JSON-friendly payload

Every `PredictionResult` can be flattened to a plain dict, which is
what HTTP handlers and message queues typically serialise.

In [49]:
import json
result = detector.predict(phish_example['sender'], phish_example['subject'], phish_example['body'])
print(json.dumps(result.to_dict(), indent=2, default=float)[:600], '...')

{
  "predicted_label": 1,
  "phishing_probability": 0.9993551740616256,
  "legitimate_probability": 0.0006448259383744492,
  "threshold": 0.75,
  "model_version": "v1_0",
  "engineered_features": {
    "body_word_count": 11.0,
    "body_exclamation_count": 0.0,
    "email_local_length": 6.0,
    "name_email_consistency": 1.0,
    "body_url_density": 9.090909090909092,
    "body_url_count": 1.0,
    "body_entropy": 4.166404998061427,
    "email_digit_ratio": 0.0,
    "domain_entropy": 3.4058222502856896,
    "domain_length": 23.0,
    "subject_entropy": 4.003701696057348,
    "body_avg_word_len ...


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


## 4. Batch prediction

`predict_batch` runs **one** vectorised forward pass — always prefer
it to looping `predict()`. It accepts a custom `threshold` and returns
a list of `PredictionResult`s in the same order as the input.

In [50]:
emails = [{k: s[k] for k in ('sender', 'subject', 'body')} for s in samples]
results = detector.predict_batch(emails, threshold=0.5)

batch_df = pd.DataFrame({
    'true_label': [s['label'] for s in samples],
    'pred_label': [r.predicted_label for r in results],
    'phish_prob': [round(r.phishing_probability, 4) for r in results],
    'subject':    [s['subject'][:60] for s in samples],
})
accuracy = float((batch_df['true_label'] == batch_df['pred_label']).mean())
print(f'batch accuracy @ threshold=0.5: {accuracy:.2%}')
batch_df

batch accuracy @ threshold=0.5: 80.00%


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,true_label,pred_label,phish_prob,subject
0,0,0,0.0056,ACM TechNews — April edition
1,1,1,0.9994,Unusual sign-in attempt detected — confirm now
2,1,1,0.9986,Package delivery failed — reschedule within 12...
3,0,0,0.0624,Recipe for that soup you liked
4,0,0,0.0169,Lunch on Saturday?
5,0,1,0.9365,Your weekly digest: 5 new repositories trending
6,0,1,0.8715,New course recommendation based on your interests
7,1,1,0.9215,Payroll update: click to review your new paystub
8,1,1,0.9996,URGENT: Your account will be suspended in 24 h...
9,1,1,0.9986,Action required: verify your identity immediately


## 5. Threshold behaviour

`phishing_probability` is fixed for a given email; `threshold` decides
only how we convert that probability to a label. Sweeping the
threshold over [0.1, 0.9] shows the flip points per sample without
re-running the model.

In [51]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
probs = np.array([r.phishing_probability for r in results])
sweep = pd.DataFrame({
    'phish_prob': probs.round(4),
    'true_label': [s['label'] for s in samples],
    **{f'pred@{t}': (probs >= t).astype(int) for t in thresholds},
})
sweep

,phish_prob,true_label,pred@0.1,pred@0.3,pred@0.5,pred@0.7,pred@0.9
0,0.0056,0,0,0,0,0,0
1,0.9994,1,1,1,1,1,1
2,0.9986,1,1,1,1,1,1
3,0.0624,0,0,0,0,0,0
4,0.0169,0,0,0,0,0,0
5,0.9365,0,1,1,1,1,1
6,0.8715,0,1,1,1,1,0
7,0.9215,1,1,1,1,1,1
8,0.9996,1,1,1,1,1,1
9,0.9986,1,1,1,1,1,1


## 6. Safe prediction — errors as data

`predict_safe` never raises. On success it returns the full
`PredictionResult.to_dict()` payload; on failure it returns an
`{'error', 'message'}` envelope, which is convenient for HTTP handlers
and batch pipelines that cannot afford exceptions.

In [52]:
valid_payload = detector.predict_safe(
    sender=phish_example['sender'],
    subject=phish_example['subject'],
    body=phish_example['body'],
    threshold=0.5,
)
print('valid   ->', {k: valid_payload[k]
                     for k in ('predicted_label', 'phishing_probability', 'threshold')})

invalid_payload = detector.predict_safe(
    sender=phish_example['sender'],
    subject=phish_example['subject'],
    body=phish_example['body'],
    threshold=1.5,  # out of range — surfaces as error-as-data
)
print('invalid ->', invalid_payload)

valid   -> {'predicted_label': 1, 'phishing_probability': 0.9993551740616256, 'threshold': 0.5}
invalid -> {'error': 'validation_error', 'message': 'threshold must be in [0, 1], got 1.5'}


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


### Same validation via the exception path

`predict` raises `ValidationError` for the same inputs — useful when
you want the error to propagate.

In [53]:
try:
    detector.predict(
        phish_example['sender'], phish_example['subject'], phish_example['body'],
        threshold=1.5,
    )
except ValidationError as e:
    print('caught ValidationError:', e)

caught ValidationError: threshold must be in [0, 1], got 1.5


## 7. Calibrated probabilities + three-zone classification

`PhishingDetector.from_paths(...)` is the escape hatch when you want
explicit control of the artefacts being loaded: calibrator, review
thresholds, and (later) a drift monitor. Loading a calibrator turns the
MLP's raw softmax outputs into probabilities that track empirical
frequencies — a score of 0.8 becomes an actual 80% confidence claim.

Setting `review_low_threshold` and `review_high_threshold` together
enables the three-zone rule:

```
prob < low             → ConfidenceZone.NOT_SPAM
low ≤ prob < high      → ConfidenceZone.REVIEW
prob ≥ high            → ConfidenceZone.SPAM
```

Both must be provided together — passing only one raises
`ValidationError`.

In [54]:
from inference import ConfidenceZone

models_root = Path(os.environ['AURA_MODELS_DIR'])
# Discover the active version via the registry (same logic load_production uses).
from inference import ModelRegistry
registry = ModelRegistry(models_root)
active = registry.active_version() or registry.latest_version()
paths = registry.paths_for(active)

calibrator_path = paths.get('calibrator')

# zoned_detector = PhishingDetector.from_paths(
#     model_path=paths['model'],
#     subject_vectorizer_path=paths['subject_vectorizer'],
#     body_vectorizer_path=paths['body_vectorizer'],
#     calibrator_path=calibrator_path,   # may be None if the version has no calibrator
#     review_low_threshold=0.3,
#     review_high_threshold=0.8,
# )

zoned_detector = PhishingDetector.from_paths(
    model_path=paths['model'],
    subject_vectorizer_path=paths['subject_vectorizer'],
    body_vectorizer_path=paths['body_vectorizer'],
    calibrator_path=calibrator_path,
    review_low_threshold=0.1,
    review_high_threshold=0.95,
)

print('calibrated :', calibrator_path is not None)
print('review_low :', zoned_detector.review_low_threshold)
print('review_high:', zoned_detector.review_high_threshold)

calibrated : True
review_low : 0.1
review_high: 0.95


### Three emails — one per zone

We pick three emails that should (heuristically) land in each zone: a
clear phish, a clear legitimate, and a borderline case with a single
suspicious-looking link. With `low=0.3` / `high=0.8`, the calibrated
probability directly selects the zone.

In [55]:
zone_cases = [
    {
        'tag': 'expected SPAM',
        'sender': '"PayPal Security" <service@paypa1-alerts.com>',
        'subject': 'URGENT: Your account will be suspended in 24 hours!!!',
        'body': 'Dear customer, we detected suspicious activity on your account. '
                'Please verify your credentials at http://paypa1-alerts.com/verify '
                'within 24 hours or your account will be locked!!!',
    },
    {
        'tag': 'expected SPAM',
        'sender': '"Chase Bank" <alerts@secure-chase-online.com>',
        'subject': 'Unusual sign-in attempt detected — confirm now',
        'body': 'Final notice: update payment details at '
                'http://secure-chase-online.com/account/update to avoid service disruption.',
    },
    {
        'tag': 'expected SPAM',
        'sender': '"Microsoft 365" <no-reply@ms-support-verify.net>',
        'subject': 'Action required: verify your identity immediately',
        'body': 'Your mailbox is over quota. Click http://ms-support-verify.net/login '
                'to re-authenticate NOW, otherwise incoming mail will bounce.',
    },
    {
        'tag': 'expected SPAM',
        'sender': '"DHL Express" <tracking@dhl-delivery-notice.info>',
        'subject': 'Package delivery failed — reschedule within 12 hours',
        'body': 'Your DHL package could not be delivered. Confirm address at '
                'http://dhl-delivery-notice.info/track?id=982137 before it is returned.',
    },
    {
        'tag': 'expected NOT_SPAM',
        'sender': '"Jane Doe" <jane.doe@gmail.com>',
        'subject': 'Lunch on Saturday?',
        'body': 'Hey, are you free for lunch Saturday around noon? Thinking of that '
                'ramen place on 5th. Let me know.',
    },
    {
        'tag': 'expected NOT_SPAM',
        'sender': '"GitHub" <noreply@github.com>',
        'subject': 'Your weekly digest: 5 new repositories trending',
        'body': 'Here are the repositories trending this week in your languages. '
                'Visit github.com/trending to see the full list.',
    },
    {
        'tag': 'expected NOT_SPAM',
        'sender': '"Mom" <mom@familymail.net>',
        'subject': 'Recipe for that soup you liked',
        'body': 'Hi sweetie, here is the lentil soup recipe I promised. Soak the '
                'lentils overnight, then simmer with carrots and celery for an hour.',
    },
    {
        'tag': 'borderline',
        'sender': '"IT Helpdesk" <helpdesk@company-support.co>',
        'subject': 'Please review the attached document',
        'body': 'Hi, we are rolling out a new policy. You can review the draft at '
                'http://company-support.co/policy-draft and let us know if anything '
                'looks off. No action required today.',
    },
    {
        'tag': 'borderline',
        'sender': '"HR Department" <hr@corp-updates.net>',
        'subject': 'Important: update your direct deposit information',
        'body': 'Please log in to the HR portal at http://corp-updates.net/payroll '
                'to confirm your banking details before the next pay cycle.',
    },
    {
        'tag': 'borderline',
        'sender': '"IT Security" <security@internalcorp.com>',
        'subject': 'Password expiry notice — action may be required',
        'body': 'Your corporate password expires in 7 days. If you wish to change it '
                'early, visit the self-service portal at internalcorp.com/password-reset. '
                'No action needed if you are happy to wait for the prompt.',
    },
    # --- New borderline / expected REVIEW emails ---
    {
        'tag': 'borderline',
        'sender': '"DocuSign" <dse@docusign.net>',
        'subject': 'Your document is ready for signature',
        'body': 'Hello, a document has been sent to you for review and signature. '
                'Please visit docusign.net/sign to review the document at your '
                'earliest convenience. This request will expire in 5 days.',
    },
    {
        'tag': 'borderline',
        'sender': '"LinkedIn" <messages-noreply@linkedin.com>',
        'subject': 'You have a new message from a recruiter',
        'body': 'Hi, a recruiter from a company in your industry has sent you a message. '
                'Log in to linkedin.com/messaging to read and respond. '
                'This is a time-sensitive opportunity.',
    },
    {
        'tag': 'borderline',
        'sender': '"Dropbox" <no-reply@dropbox.com>',
        'subject': 'Someone shared a folder with you',
        'body': 'A folder has been shared with you on Dropbox. '
                'Visit dropbox.com/sh/folder to view the contents. '
                'You will need to log in or create an account to access the files.',
    },
    {
        'tag': 'borderline',
        'sender': '"Microsoft" <microsoft-noreply@microsoft.com>',
        'subject': 'Unusual sign-in activity on your account',
        'body': 'We noticed a sign-in to your Microsoft account from a new device. '
                'If this was you, no action is needed. If you do not recognise this '
                'activity, please review your account at account.microsoft.com/security.',
    },
    {
        'tag': 'borderline',
        'sender': '"Accounts Payable" <ap@vendor-invoices.net>',
        'subject': 'Invoice #8821 — payment due in 3 days',
        'body': 'Please find attached invoice #8821 for services rendered in March. '
                'Total amount due: $4,250.00. Please remit payment to the account '
                'details on file or contact us at ap@vendor-invoices.net.',
    },
    {
        'tag': 'borderline',
        'sender': '"Google" <no-reply@accounts.google.com>',
        'subject': 'Security alert — new sign-in on Chrome',
        'body': 'Your Google account was just signed in to on a Windows device. '
                'If this was you, you can ignore this message. '
                'If not, visit myaccount.google.com/security to secure your account.',
    },
    {
        'tag': 'borderline',
        'sender': '"Zoom" <no-reply@zoom.us>',
        'subject': 'Your Zoom meeting recording is now available',
        'body': 'The recording from your meeting on April 15 is now ready. '
                'You can view and download it at zoom.us/recording. '
                'Recordings are stored for 30 days before being deleted.',
    },
    {
        'tag': 'borderline',
        'sender': '"Sarah Johnson" <s.johnson@acme-corp.net>',
        'subject': 'Quick favour — can you approve this today?',
        'body': 'Hi, I know this is short notice but I need approval on a vendor '
                'payment before end of day. The details are in the attached file. '
                'Please let me know if you can action this. Thanks, Sarah.',
    },
    {
        'tag': 'borderline',
        'sender': '"Stripe" <support@stripe.com>',
        'subject': 'Action required — update your payout details',
        'body': 'We were unable to process your recent payout due to an issue with '
                'your bank account details. Please log in to your Stripe dashboard at '
                'dashboard.stripe.com to update your information and avoid payout delays.',
    },
    {
        'tag': 'borderline',
        'sender': '"IT Support" <support@internal-helpdesk.com>',
        'subject': 'Scheduled maintenance — action required before Friday',
        'body': 'We are migrating email accounts to a new server this Friday. '
                'To ensure continuity please log in to the IT portal at '
                'internal-helpdesk.com/migrate and confirm your account details '
                'before Thursday evening.',
    },
]

zone_rows = []
for case in zone_cases:
    r = zoned_detector.predict(case['sender'], case['subject'], case['body'])
    zone_rows.append({
        'tag': case['tag'],
        'phish_prob': round(r.phishing_probability, 4),
        'confidence_zone': r.confidence_zone.value if r.confidence_zone else None,
        'predicted_label': r.predicted_label,
        'calibrated': r.calibrated,
        'prediction_id': r.prediction_id,
    })

pd.DataFrame(zone_rows)

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,tag,phish_prob,confidence_zone,predicted_label,calibrated,prediction_id
0,expected SPAM,0.9996,SPAM,1,True,8b511f81-853c-4eee-b2f3-1edea265ef95
1,expected SPAM,0.9994,SPAM,1,True,85a45a87-27f9-467d-bdef-9b2e3c6e144f
2,expected SPAM,0.9986,SPAM,1,True,c3a19ca6-85bd-45ae-b9ad-b96319d125ba
3,expected SPAM,0.9986,SPAM,1,True,57507b6c-0c4c-4d33-8722-480649b66840
4,expected NOT_SPAM,0.0169,NOT_SPAM,0,True,9ec3acc0-a947-4b2b-8643-a7a2c96410a4
5,expected NOT_SPAM,0.9365,REVIEW,1,True,edcfdbf1-033f-4651-9bc4-58281cc4133c
6,expected NOT_SPAM,0.0624,NOT_SPAM,0,True,a7cbf83d-f59a-45ea-a62d-a04591272c92
7,borderline,0.2631,REVIEW,0,True,ea78270a-9c91-4bcd-97e3-9751bf215ff6
8,borderline,0.9824,SPAM,1,True,25e8674f-c442-450d-9bdf-be40e6bff6ca
9,borderline,0.8664,REVIEW,1,True,36b3d534-5424-4340-ae48-19acfd908d25


## 8. `prediction_id` — pairing predictions with confirmations

Every result carries a UUID4 `prediction_id`. This is the field that
lets you tie a prediction to a later ground-truth signal — a user-report
button, a downstream sandbox verdict, or a human reviewer — without
making any assumption about how those signals get back to the service.

`DriftMonitor.record_confirmation(prediction_id, confirmed_label=...)`
consumes exactly this id. See `demo_drift_monitor.ipynb` for the full
loop; here we simply show that the id is stable across `.to_dict()`
and safe to serialise.

In [56]:
r = zoned_detector.predict(
    zone_cases[0]['sender'], zone_cases[0]['subject'], zone_cases[0]['body'],
)
payload = r.to_dict()

print('prediction_id  :', r.prediction_id)
print('in to_dict()   :', payload['prediction_id'])
print('JSON-safe round-trip:', json.loads(json.dumps(payload, default=float))['prediction_id'])
print('confidence_zone serialised as plain string:', type(payload['confidence_zone']).__name__,
      '=', payload['confidence_zone'])

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


prediction_id  : 1c7b3409-eebc-4e47-91d1-741da6afe614
in to_dict()   : 1c7b3409-eebc-4e47-91d1-741da6afe614
JSON-safe round-trip: 1c7b3409-eebc-4e47-91d1-741da6afe614
confidence_zone serialised as plain string: str = SPAM
